# WAF Reliability
- Assessment Audit

This notebook assesses the Reliability pillar using the **4 Key Topics** structure.

## Scoring Model (1-5 Scale)
| Score | Rating | Description |
|-------|--------|-------------|
| 5 | Excellent | Best practices fully implemented |
| 4 | Good | Most best practices, minor gaps |
| 3 | Moderate | Basic implementation |
| 2 | Limited | Minimal implementation |
| 1 | Poor | Not implemented |

## 4 Key Topics

1. **Governance**
- SLA Definition, Ownership Model, Operational Procedures, Change Management

2. **Tolerance**
- Retry Logic, Idempotent Operations, Graceful Degradation, Circuit Breakers

3. **Resiliency**
- Multi-Region Deployment, Client Redirect, Workload Isolation, Load Distribution

4. **Recovery**
- Time Travel, Fail-safe, Backup Strategy, Disaster Recovery Testing

In [1]:
SCHEMA = "IE"
ACCOUNT_ID = 100058
DATABASE = "SNOWHOUSE_IMPORT"
print(f"Target: {DATABASE}.{SCHEMA} | Account ID: {ACCOUNT_ID}")

Target: SNOWHOUSE_IMPORT.IE | Account ID: 100058


In [2]:
from snowflake.snowpark import Session
import pandas as pd
import os

connection_name = os.getenv("SNOWFLAKE_CONNECTION_NAME") or "snowhouse"
if 'session' not in globals():
    session = Session.builder.config("connection_name", connection_name).create()
print(f"Connected via: {connection_name}")

/Users/edscott/repos/tam-main/.venv/lib/python3.11/site-packages/snowflake/connector/config_manager.py:369: UserWarning: Bad owner or permissions on /Users/edscott/.snowflake/connections.toml.
 * To change owner, run `chown $USER "/Users/edscott/.snowflake/connections.toml"`.
 * To restrict permissions, run `chmod 0600 "/Users/edscott/.snowflake/connections.toml"`.
 * To skip this warning, set environment variable SF_SKIP_WARNING_FOR_READ_PERMISSIONS_ON_CONFIG_FILE=true.

  warn(f"Bad owner or permissions on {str(filep)}{chmod_message}")
 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://snowbiz.okta.com/app/snowflake/exk8wfsfryJIn4IWZ2p7/sso/saml?SAMLRequest=jZJBc9owEIX%2Fikc9Y9luaECDyTjQTJymhgkQOtyELRsVWXK1Mob8%2BsomdNJDMrlpVu9J3%2B7b0c2xFM6BaeBKhsh3PeQwmaqMyyJEq%2BVdb4AcMFRmVCjJQnRigG7GI6ClqEhUm518Yn9qBsaxD0kg3UWIai2JosCBSFoyICYli%2BjnIwlcj1RaGZUqgd5YPnZQAKaNJbxYMuAWb2dMRTBumsZtvrpKFzjwPA97Q2xVreTLRX%2B0Pb2j97F31eqtwsrnr2y3XJ5H8BHW9iwCcr9cznvz2WKJnOiCOlES6pLpBdMHnrLV0%2BMZACzBIpmt72erxXcXpGpyQfcsVWVVG%2Fuaa084ZxkWquC24XgaomrPs5M5mN%2Bzh%2B2PopB1%2BXK7PvpBsJkk4lc5qHb9Jkq2yZardTKEFDnPl0SDNtEYoGaxbHM0tuQF33qe3wuGS98n%2FpAE127f72%2BQM7U5cklN57zAtohb%2FuKqvaEdHK0q%2FI8bs%2BN%2B0OSQ69NDLK%2Fi9SaorjGAwm2s6LwppAPQ48%2F2P8JvXa%2FLltj5x9O5Ejw9OXdKl9S8H4%2Fv%2Bl2FZ728kxJWUi6iLNMMwMYkhGommlFjd9romiE8Pv%2F6%2F1aP%2FwI%3D&RelayState=ver%3A1-hint%3A2051115165194-ETMsDgAAAZwJesgbABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEFCIqg85IQ7F0A5%2Fe8jODOoAAACwBGyCOvDo279tXVHI9V%2Fb2F

 pip install snowflake-connector-python[secure-local-storage]


---
# Key Topic 1: GOVERNANCE

**Establish reliability standards and ownership.**

| Sub-Topic | DDM | Manual |
|-----------|-----|--------|
| SLA Definition | No | Yes |
| Ownership Model | Partial | Yes |
| Operational Procedures | No | Yes |
| Change Management | Partial | Yes |

## Manual Assessment: SLA Definition

**Questions:**
1. SLAs (RTO/RPO) defined for critical workloads?
2. Workloads tiered by criticality?
3. SLAs reviewed periodically?

**Scoring:** 5=Documented SLAs+tiered | 3=Some SLAs | 1=None defined

**Manual Score:** _____

In [3]:
# Ownership Model - Check for ownership tags
tags_df = session.sql(f"""
SELECT COUNT(CASE WHEN UPPER(NAME) LIKE '%OWNER%' OR UPPER(NAME) LIKE '%TEAM%' THEN ID END) as ownership_tags
FROM {DATABASE}.{SCHEMA}.TAG_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
own_tags = tags_df['OWNERSHIP_TAGS'].iloc[0]
print(f"=== Ownership Model ===\nOwnership Tags: {own_tags}")
ownership_ddm_score = 5 if own_tags >= 3 else 4 if own_tags >= 2 else 3 if own_tags >= 1 else 1
print(f"DDM Score: {ownership_ddm_score}/5")

=== Ownership Model ===
Ownership Tags: 1
DDM Score: 3/5


## Manual Assessment: Ownership Model

**Questions:**
1. Clear ownership for data assets?
2. On-call rotation defined?
3. Escalation paths documented?

**Scoring:** 5=Clear ownership+on-call | 3=Informal ownership | 1=No ownership

**Manual Score:** _____

## Manual Assessment: Operational Procedures

**Questions:**
1. Runbooks documented?
2. Escalation paths defined?
3. Incident response tested?

**Scoring:** 5=Comprehensive runbooks | 3=Basic documentation | 1=None

**Manual Score:** _____

In [4]:
# Change Management - Check Time Travel settings
tt_df = session.sql(f"""
SELECT COUNT(*) as total_dbs,
    COUNT(CASE WHEN DATA_TRANSIENT = FALSE THEN 1 END) as non_transient
FROM {DATABASE}.{SCHEMA}.DATABASE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
tt_pct = (tt_df['NON_TRANSIENT'].iloc[0] / tt_df['TOTAL_DBS'].iloc[0] * 100) if tt_df['TOTAL_DBS'].iloc[0] > 0 else 0
print(f"=== Change Management ===\nNon-transient DBs: {tt_pct:.1f}%")
change_ddm_score = 5 if tt_pct >= 90 else 4 if tt_pct >= 70 else 3 if tt_pct >= 50 else 2
print(f"DDM Score: {change_ddm_score}/5")

=== Change Management ===
Non-transient DBs: 0.0%
DDM Score: 2/5


## Manual Assessment: Change Management

**Questions:**
1. Approval workflows in place?
2. Rollback procedure documented?
3. Changes tested in lower environments?

**Scoring:** 5=Full approval process | 3=Informal process | 1=No process

**Manual Score:** _____

In [5]:
sla_score = 3; ownership_manual = 3; ops_score = 3; change_manual = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 1: GOVERNANCE - Summary\n" + "="*60)
print(f"  SLA Definition (Manual): {sla_score}/5\n  Ownership (DDM): {ownership_ddm_score}/5\n  Ownership (Manual): {ownership_manual}/5")
print(f"  Operational Procedures (Manual): {ops_score}/5\n  Change Management (DDM): {change_ddm_score}/5\n  Change Management (Manual): {change_manual}/5")
governance_score = round((sla_score + ownership_ddm_score + ownership_manual + ops_score + change_ddm_score + change_manual) / 6, 1)
print(f"\n📊 GOVERNANCE TOPIC SCORE: {governance_score}/5")

KEY TOPIC 1: GOVERNANCE - Summary
  SLA Definition (Manual): 3/5
  Ownership (DDM): 3/5
  Ownership (Manual): 3/5
  Operational Procedures (Manual): 3/5
  Change Management (DDM): 2/5
  Change Management (Manual): 3/5

📊 GOVERNANCE TOPIC SCORE: 2.8/5


---
# Key Topic 2: TOLERANCE

**Handle expected failures gracefully.**

| Sub-Topic | DDM | Manual |
|-----------|-----|--------|
| Retry Logic | Partial | Yes |
| Idempotent Operations | No | Yes |
| Graceful Degradation | No | Yes |
| Circuit Breakers | No | Yes |

In [6]:
# Retry Logic - Check task retry settings
task_df = session.sql(f"""
SELECT COUNT(*) as total_tasks,
    COUNT(CASE WHEN ALLOW_OVERLAPPING_EXECUTION = FALSE THEN 1 END) as safe_tasks
FROM {DATABASE}.{SCHEMA}.USER_TASK_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
task_count = task_df['TOTAL_TASKS'].iloc[0]
print(f"=== Retry Logic ===\nTasks configured: {task_count}")
retry_ddm_score = 5 if task_count >= 10 else 4 if task_count >= 5 else 3 if task_count >= 1 else 1
print(f"DDM Score: {retry_ddm_score}/5")

=== Retry Logic ===
Tasks configured: 5527
DDM Score: 5/5


## Manual Assessment: Retry Logic

**Questions:**
1. Exponential backoff implemented?
2. Transient errors handled?
3. Max retries configured?

**Scoring:** 5=Standardized retry patterns | 3=Some retry logic | 1=No retry logic

**Manual Score:** _____

## Manual Assessment: Idempotent Operations

**Questions:**
1. Pipelines designed for re-run safety?
2. MERGE statements used for upserts?
3. Duplicate handling in place?

**Scoring:** 5=All operations idempotent | 3=Critical operations only | 1=Not considered

**Manual Score:** _____

## Manual Assessment: Graceful Degradation

**Questions:**
1. Systems continue with reduced capability on failure?
2. Fallback mechanisms in place?
3. Degraded mode tested?

**Scoring:** 5=Comprehensive fallbacks | 3=Some fallbacks | 1=No fallbacks

**Manual Score:** _____

## Manual Assessment: Circuit Breakers

**Questions:**
1. Circuit breaker patterns implemented?
2. Failure thresholds defined?
3. Auto-recovery configured?

**Scoring:** 5=Implemented+tested | 3=Partial implementation | 1=Not implemented

**Manual Score:** _____

In [7]:
retry_manual = 3; idempotent_score = 3; degradation_score = 3; circuit_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 2: TOLERANCE - Summary\n" + "="*60)
print(f"  Retry Logic (DDM): {retry_ddm_score}/5\n  Retry Logic (Manual): {retry_manual}/5")
print(f"  Idempotent (Manual): {idempotent_score}/5\n  Graceful Degradation (Manual): {degradation_score}/5\n  Circuit Breakers (Manual): {circuit_score}/5")
tolerance_score = round((retry_ddm_score + retry_manual + idempotent_score + degradation_score + circuit_score) / 5, 1)
print(f"\n📊 TOLERANCE TOPIC SCORE: {tolerance_score}/5")

KEY TOPIC 2: TOLERANCE - Summary
  Retry Logic (DDM): 5/5
  Retry Logic (Manual): 3/5
  Idempotent (Manual): 3/5
  Graceful Degradation (Manual): 3/5
  Circuit Breakers (Manual): 3/5

📊 TOLERANCE TOPIC SCORE: 3.4/5


---
# Key Topic 3: RESILIENCY

**Architect for high availability.**

| Sub-Topic | DDM | Source |
|-----------|-----|--------|
| Multi-Region Deployment | Yes | REPLICATION_GROUP_ETL_V |
| Client Redirect | Partial | REPLICATION_GROUP_ETL_V |
| Workload Isolation | Yes | WAREHOUSE_ETL_V |
| Load Distribution | Yes | WAREHOUSE_ETL_V (MCW) |

In [8]:
# Multi-Region Deployment
rep_df = session.sql(f"""
SELECT COUNT(*) as rep_groups,
    COUNT(CASE WHEN IS_FAILOVER_GROUP THEN 1 END) as failover_groups
FROM {DATABASE}.{SCHEMA}.REPLICATION_GROUP_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
rep_count = rep_df['REP_GROUPS'].iloc[0]
failover_count = rep_df['FAILOVER_GROUPS'].iloc[0]
print(f"=== Multi-Region ===\nReplication Groups: {rep_count} | Failover Groups: {failover_count}")
multi_score = 5 if failover_count >= 2 else 4 if failover_count >= 1 else 3 if rep_count >= 1 else 1
print(f"Score: {multi_score}/5")

=== Multi-Region ===
Replication Groups: 9 | Failover Groups: 0
Score: 3/5


In [9]:
# Workload Isolation
wh_df = session.sql(f"""
SELECT COUNT(DISTINCT NAME) as wh_count
FROM {DATABASE}.{SCHEMA}.WAREHOUSE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL
""").to_pandas()
wh_count = wh_df['WH_COUNT'].iloc[0]
print(f"=== Workload Isolation ===\nWarehouses: {wh_count}")
isolation_score = 5 if wh_count >= 10 else 4 if wh_count >= 5 else 3 if wh_count >= 3 else 2
print(f"Score: {isolation_score}/5")

=== Workload Isolation ===
Warehouses: 157
Score: 5/5


In [10]:
# Load Distribution (Multi-Cluster)
mcw_df = session.sql(f"""
SELECT COUNT(*) as mcw_count
FROM {DATABASE}.{SCHEMA}.WAREHOUSE_ETL_V
WHERE ACCOUNT_ID = {ACCOUNT_ID} AND DELETED_ON IS NULL AND MAX_CLUSTER_COUNT > 1
""").to_pandas()
mcw_count = mcw_df['MCW_COUNT'].iloc[0]
print(f"=== Load Distribution ===\nMulti-cluster Warehouses: {mcw_count}")
load_score = 5 if mcw_count >= 5 else 4 if mcw_count >= 3 else 3 if mcw_count >= 1 else 1
print(f"Score: {load_score}/5")

=== Load Distribution ===
Multi-cluster Warehouses: 153
Score: 5/5


In [11]:
print("="*60 + "\nKEY TOPIC 3: RESILIENCY - Summary\n" + "="*60)
print(f"  Multi-Region: {multi_score}/5\n  Workload Isolation: {isolation_score}/5\n  Load Distribution: {load_score}/5")
resiliency_score = round((multi_score + isolation_score + load_score) / 3, 1)
print(f"\n📊 RESILIENCY TOPIC SCORE: {resiliency_score}/5")

KEY TOPIC 3: RESILIENCY - Summary
  Multi-Region: 3/5
  Workload Isolation: 5/5
  Load Distribution: 5/5

📊 RESILIENCY TOPIC SCORE: 4.3/5


---
# Key Topic 4: RECOVERY

**Restore operations quickly after failures.**

| Sub-Topic | DDM | Manual |
|-----------|-----|--------|
| Time Travel | Yes |
- |
| Fail-safe | Partial | Yes |
| Backup Strategy | Partial | Yes |
| Disaster Recovery Testing | No | Yes |

In [12]:
# Time Travel
print(f"=== Time Travel ===\nNon-transient DBs: {tt_pct:.1f}%")
tt_score = 5 if tt_pct >= 90 else 4 if tt_pct >= 70 else 3 if tt_pct >= 50 else 2 if tt_pct >= 20 else 1
print(f"Score: {tt_score}/5")

=== Time Travel ===
Non-transient DBs: 0.0%
Score: 1/5


## Manual Assessment: Fail-safe

**Questions:**
1. Fail-safe understood and documented?
2. 7-day recovery window known?
3. Support ticket process established?

**Scoring:** 5=Documented+tested | 3=Understood | 1=Not considered

**Manual Score:** _____

## Manual Assessment: Backup Strategy

**Questions:**
1. External data export for backup?
2. Backup frequency defined?
3. Backup testing performed?

**Scoring:** 5=Regular backups+tested | 3=Some backups | 1=No backups

**Manual Score:** _____

## Manual Assessment: DR Testing

**Questions:**
1. Failover tested quarterly?
2. Recovery times validated?
3. DR runbooks documented?

**Scoring:** 5=Regular testing+documented | 3=Occasional testing | 1=Never tested

**Manual Score:** _____

In [13]:
failsafe_score = 3; backup_score = 3; dr_score = 3  # Update from manual
print("="*60 + "\nKEY TOPIC 4: RECOVERY - Summary\n" + "="*60)
print(f"  Time Travel (DDM): {tt_score}/5\n  Fail-safe (Manual): {failsafe_score}/5")
print(f"  Backup Strategy (Manual): {backup_score}/5\n  DR Testing (Manual): {dr_score}/5")
recovery_score = round((tt_score + failsafe_score + backup_score + dr_score) / 4, 1)
print(f"\n📊 RECOVERY TOPIC SCORE: {recovery_score}/5")

KEY TOPIC 4: RECOVERY - Summary
  Time Travel (DDM): 1/5
  Fail-safe (Manual): 3/5
  Backup Strategy (Manual): 3/5
  DR Testing (Manual): 3/5

📊 RECOVERY TOPIC SCORE: 2.5/5


---
# Overall Assessment Summary

In [14]:
print("="*70 + "\nRELIABILITY - OVERALL ASSESSMENT\n" + "="*70)
topics = {"1. Governance": governance_score, "2. Tolerance": tolerance_score, "3. Resiliency": resiliency_score, "4. Recovery": recovery_score}
for topic, score in topics.items():
    bar = "█" * int(score) + "░" * (5 - int(score))
    status = "✅" if score >= 4 else "⚠️" if score >= 3 else "❌"
    print(f"{status} {topic}: {bar} {score}/5")
overall_score = round(sum(topics.values()) / len(topics), 1)
print(f"\n🏆 OVERALL RELIABILITY SCORE: {overall_score}/5")

RELIABILITY - OVERALL ASSESSMENT
❌ 1. Governance: ██░░░ 2.8/5
⚠️ 2. Tolerance: ███░░ 3.4/5
✅ 3. Resiliency: ████░ 4.3/5
❌ 4. Recovery: ██░░░ 2.5/5

🏆 OVERALL RELIABILITY SCORE: 3.2/5


In [15]:
prompt = f"""WAF Reliability advisor. Provide:
1. EXECUTIVE SUMMARY (2-3 sentences)
2. CRITICAL FINDINGS (3-5 issues with priority)
3. QUICK WINS (3-5 actions for 1 week)

Reliability: {overall_score}/5 | Topics: {topics}"""
result = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-sonnet', '{prompt.replace(chr(39), chr(39)+chr(39))}') as response").collect()
print("🤖 AI INSIGHTS\n" + result[0]['RESPONSE'])

🤖 AI INSIGHTS
1. EXECUTIVE SUMMARY
The WAF implementation shows moderate reliability with significant room for improvement, particularly in governance and recovery areas. While resiliency measures are relatively strong (4.3/5), the overall reliability score of 3.2/5 indicates immediate attention is needed to strengthen the WAF's operational stability and incident response capabilities.

2. CRITICAL FINDINGS
Priority 1: Weak governance framework (2.8/5) - Insufficient policies and procedures for WAF management
Priority 2: Inadequate recovery mechanisms (2.5/5) - Poor backup and failover processes
Priority 3: Moderate tolerance levels (3.4/5) - Need for improved error handling and request processing
Priority 4: Documentation gaps across operational procedures and incident response plans

3. QUICK WINS
1. Implement automated backup verification checks (24-48 hours)
2. Create basic incident response runbooks for top 3 common WAF issues (3-4 days)
3. Enable enhanced logging and monitoring f

---
## Coverage Matrix

| Topic | Sub-Topic | DDM | Manual | Source |
|-------|-----------|-----|--------|--------|
| Governance | SLA Definition |
- | ✅ | N/A |
| Governance | Ownership Model | ✅ | ✅ | TAG_ETL_V |
| Governance | Operational Procedures |
- | ✅ | N/A |
| Governance | Change Management | ✅ | ✅ | DATABASE_ETL_V |
| Tolerance | Retry Logic | ✅ | ✅ | USER_TASK_ETL_V |
| Tolerance | Idempotent Operations |
- | ✅ | N/A |
| Tolerance | Graceful Degradation |
- | ✅ | N/A |
| Tolerance | Circuit Breakers |
- | ✅ | N/A |
| Resiliency | Multi-Region | ✅ |
- | REPLICATION_GROUP_ETL_V |
| Resiliency | Workload Isolation | ✅ |
- | WAREHOUSE_ETL_V |
| Resiliency | Load Distribution | ✅ |
- | WAREHOUSE_ETL_V |
| Recovery | Time Travel | ✅ |
- | DATABASE_ETL_V |
| Recovery | Fail-safe |
- | ✅ | N/A |
| Recovery | Backup Strategy |
- | ✅ | N/A |
| Recovery | DR Testing |
- | ✅ | N/A |

**Summary:** 8 DDM | 3 Partial | 6 Manual Only